# Lesson 2a: Multilayer Perceptrons and Backpropagation — Theory

Lessons 1a/1b showed that a single neuron (logistic regression) learns by
gradient descent on a smooth loss, and that its gradient collapses to the
remarkably simple error signal $(\hat y - y)$. But a single neuron is still
just a linear decision boundary wrapped in one non-linearity — lesson 0a's
XOR problem already proved that some problems need more than one neuron.

This notebook stacks neurons into **layers**, and layers into a **multilayer
perceptron (MLP)**, then asks the question 1a's single neuron never had to
answer: if a prediction depends on parameters buried three layers deep
inside the network, how do you compute *its* gradient? The answer is
**backpropagation** — repeated application of the chain rule, propagating an
error signal backward from the output layer to every weight in the network.

By the end of this notebook you will have:
- derived the **forward pass** of an $L$-layer MLP in matrix form,
- derived **backpropagation** from the chain rule, layer by layer, arriving
  at a clean recursive formula for every weight and bias gradient,
- implemented the forward and backward passes **from scratch in NumPy** — no
  autograd — as a small MLP class,
- verified the from-scratch analytic gradients against **numerical
  finite-difference gradients**,
- trained that from-scratch MLP on a real MNIST subset and watched it learn, and
- compared **sigmoid, tanh, and ReLU** and explained why saturating
  activations make deep networks harder to train.

## Introduction

Recall the XOR problem from Lesson 0a: no single straight line
separates the two classes, so a single neuron — however its weights are
tuned — cannot solve it. The fix used there was to hand-combine two linear
boundaries through a non-linearity into a small two-layer network, with
weights chosen by hand.

A **multilayer perceptron** is that same idea, generalized and made
learnable: stack several layers of neurons, each layer's output feeding the
next layer's input, with a non-linear activation between every pair of
layers. The first $L-1$ layers ("hidden layers") build up an increasingly
useful internal representation of the input; the last layer ("output layer")
turns that representation into a prediction — here, a probability
distribution over the ten MNIST digit classes.

The forward computation of an MLP is a direct, mechanical stack of the
single-neuron computation from 1a, applied layer after layer. The genuinely
new problem this lesson solves is *training*: with a single neuron, the loss
depended on the weights directly, and one application of the chain rule gave
the gradient. With many stacked layers, the loss depends on a layer-1 weight
only *indirectly*, through everything computed after it. Backpropagation is
nothing more than the chain rule applied carefully, layer by layer, so that
this indirect dependency becomes a tractable, efficiently-computable
gradient for every parameter in the network — no matter how deep the stack.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, data subsampling) is reproducible. Seed both numpy and torch (torch is
# used only to fetch MNIST via torchvision — training itself is pure NumPy).
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

## Forward Pass

### Notation

An MLP with $L$ layers (we will use $L=3$ concretely: two hidden layers plus
one output layer, though every formula below holds for any $L$) computes, for
an input column vector $a^{[0]} = x \in \mathbb{R}^{n_0}$:

$$
z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}, \qquad a^{[l]} = g^{[l]}\big(z^{[l]}\big), \qquad l = 1, \dots, L
$$

where $W^{[l]} \in \mathbb{R}^{n_l \times n_{l-1}}$ and
$b^{[l]} \in \mathbb{R}^{n_l}$ are the weight matrix and bias of layer $l$,
and $g^{[l]}$ is that layer's activation function, applied elementwise. Each
row of $W^{[l]}$ belongs to one neuron in layer $l$: $z^{[l]}_j$ is exactly
the single-neuron weighted-sum-plus-bias computation from 1a, applied to the
*previous layer's output* instead of the raw input.

For the hidden layers ($l < L$) we use a non-linear activation such as ReLU,
tanh, or sigmoid (compared later in this notebook). For the output layer
($l=L$), classifying into 10 digit classes is a *multi-class* problem, so we
generalize 1a's sigmoid (which produces one probability) to the **softmax**,
which produces a full probability distribution over $K$ classes:

$$
a^{[L]}_k = \operatorname{softmax}(z^{[L]})_k = \frac{e^{z^{[L]}_k}}{\sum_{j=1}^{K} e^{z^{[L]}_j}}, \qquad k = 1, \dots, K
$$

Every $a^{[L]}_k \in (0,1)$ and $\sum_k a^{[L]}_k = 1$, so $a^{[L]}$ is
interpreted as $P(\text{class} = k \mid x)$ — exactly the multi-class
generalization of 1a's $\hat y = \sigma(z) = P(y=1\mid x)$.

### Loss

With one-hot true label $y \in \{0,1\}^K$ (so $y_k = 1$ for the true class,
0 otherwise) and $n$ training examples, the natural multi-class
generalization of 1a's binary cross-entropy is **categorical cross-entropy**:

$$
J = -\frac{1}{n}\sum_{i=1}^n \sum_{k=1}^{K} y^{(i)}_k \log a^{[L],(i)}_k
$$

For a batch, we stack examples as columns: $A^{[0]} \in \mathbb{R}^{n_0
\times n}$, $Z^{[l]}, A^{[l]} \in \mathbb{R}^{n_l \times n}$, and the bias is
broadcast across the $n$ columns. The forward pass is then $L$ matrix
multiplications, each followed by an elementwise (or, for softmax,
column-wise) non-linearity — a direct stack of the single-neuron
computation, one layer deep at a time.

## Deriving Backpropagation

We need $\partial J/\partial W^{[l]}$ and $\partial J/\partial
b^{[l]}$ for *every* layer $l$, including layers buried deep inside the
network where $J$ depends on them only through everything computed
afterward. The trick — backpropagation — is to define one intermediate
quantity per layer, the **error signal**

$$
\delta^{[l]} \;\equiv\; \frac{\partial J}{\partial z^{[l]}}
$$

and show that $\delta^{[l]}$ can be computed *recursively*, from the output
layer backward, reusing work already done for layer $l+1$ to get layer $l$.
Once we have $\delta^{[l]}$ for every layer, the weight and bias gradients
fall out in one more chain-rule step each.

### Step 1 — the output layer error, $\delta^{[L]}$

This is the direct multi-class generalization of 1a's Step 1–3 (sigmoid +
binary cross-entropy). For one example, with $a_k = a^{[L]}_k =
\operatorname{softmax}(z^{[L]})_k$:

**Softmax Jacobian.** Differentiating the softmax definition (quotient rule,
being careful that the denominator depends on *every* $z_j$, not just
$z_i$) gives:

$$
\frac{\partial a_k}{\partial z_i} = a_k(\mathbb{1}[i=k] - a_i)
$$

**Chain rule through cross-entropy**, using $\partial J/\partial a_k =
-y_k/a_k$:

$$
\frac{\partial J}{\partial z_i} = \sum_{k=1}^K \frac{\partial J}{\partial a_k}\cdot\frac{\partial a_k}{\partial z_i}
= \sum_{k=1}^K \Big(-\frac{y_k}{a_k}\Big)\, a_k(\mathbb{1}[i=k]-a_i)
= -\sum_{k=1}^K y_k(\mathbb{1}[i=k]-a_i)
$$

Distributing the sum and using $\sum_k y_k = 1$ (one-hot labels sum to 1):

$$
\frac{\partial J}{\partial z_i} = -y_i + a_i\underbrace{\sum_k y_k}_{=1} = a_i - y_i
$$

So, in vector form for the whole output layer:

$$
\boxed{\delta^{[L]} = a^{[L]} - y}
$$

This is *exactly* the same cancellation 1a found for sigmoid + binary
cross-entropy — softmax + categorical cross-entropy is its multi-class
twin, and the messy softmax-Jacobian and cross-entropy-derivative terms
cancel down to the same beautifully simple "predicted probability minus true
label" error signal.

### Step 2 — propagating the error backward, $\delta^{[l]}$ for $l < L$

For a hidden layer, $J$ depends on $z^{[l]}_j$ only through $a^{[l]}_j =
g(z^{[l]}_j)$, which feeds into *every* unit of the next layer via $z^{[l+1]}
= W^{[l+1]} a^{[l]} + b^{[l+1]}$. The chain rule must sum over all of those
downstream paths:

$$
\delta^{[l]}_j = \frac{\partial J}{\partial z^{[l]}_j}
= \sum_{k} \frac{\partial J}{\partial z^{[l+1]}_k}\cdot\frac{\partial z^{[l+1]}_k}{\partial a^{[l]}_j}\cdot\frac{\partial a^{[l]}_j}{\partial z^{[l]}_j}
= \Big(\sum_k \delta^{[l+1]}_k\, W^{[l+1]}_{kj}\Big)\, g'\big(z^{[l]}_j\big)
$$

since $\partial z^{[l+1]}_k/\partial a^{[l]}_j = W^{[l+1]}_{kj}$ directly from
the linear layer. The sum over $k$ is exactly $\big[(W^{[l+1]})^\top
\delta^{[l+1]}\big]_j$, so in vector form:

$$
\boxed{\delta^{[l]} = \big(W^{[l+1]}\big)^\top \delta^{[l+1]} \;\odot\; g'\big(z^{[l]}\big)}
$$

($\odot$ is the elementwise product.) This is the recursive step: given the
error signal one layer downstream, we get the error signal one layer
upstream by (1) projecting it back through the transpose of that layer's
weight matrix and (2) multiplying elementwise by the local activation
derivative. Applying Step 1 once (for $l=L$) and then Step 2 repeatedly
($l = L-1, L-2, \dots, 1$) produces every layer's error signal — this
backward sweep is why the algorithm is called **backpropagation**.

### Step 3 — from error signals to parameter gradients

The last piece is the easiest: $z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}$, so
$\partial z^{[l]}_j/\partial W^{[l]}_{jk} = a^{[l-1]}_k$ and $\partial
z^{[l]}_j/\partial b^{[l]}_j = 1$. Chaining through $\delta^{[l]}_j =
\partial J/\partial z^{[l]}_j$:

$$
\frac{\partial J}{\partial W^{[l]}_{jk}} = \delta^{[l]}_j\, a^{[l-1]}_k
\quad\Longrightarrow\quad
\boxed{\nabla_{W^{[l]}} J = \delta^{[l]} \big(a^{[l-1]}\big)^\top}
\qquad\qquad
\boxed{\nabla_{b^{[l]}} J = \delta^{[l]}}
$$

For a batch of $n$ examples (columns stacked as $A^{[l-1]}$, $\Delta^{[l]}$),
average over the batch: $\nabla_{W^{[l]}} J = \frac{1}{n}\Delta^{[l]}
(A^{[l-1]})^\top$ and $\nabla_{b^{[l]}} J = \frac{1}{n}\sum_i \Delta^{[l]}_{:,i}$
(sum across columns, i.e. across examples).

**Putting it together**, training an $L$-layer MLP is: one forward pass
(Section "Forward Pass") to get every $a^{[l]}$; one backward pass computing
$\delta^{[L]} = a^{[L]} - y$ and then $\delta^{[l]} = (W^{[l+1]})^\top
\delta^{[l+1]} \odot g'(z^{[l]})$ for $l = L-1, \dots, 1$; and finally
$\nabla_{W^{[l]}} J = \frac{1}{n}\Delta^{[l]}(A^{[l-1]})^\top$,
$\nabla_{b^{[l]}} J = \frac{1}{n}\sum \Delta^{[l]}$ for every layer, followed
by a gradient descent step $W^{[l]} \leftarrow W^{[l]} - \eta\nabla_{W^{[l]}}J$
(and likewise for $b^{[l]}$). Note that the *cost* of the backward pass is
the same order as the forward pass — one matrix multiply per layer, run in
reverse — which is what makes training deep networks computationally
feasible at all.

## Implementation from Scratch

We now implement exactly the equations above — forward pass, the
three backprop steps, and the parameter gradients — as a small NumPy `MLP`
class. There is no autograd anywhere in this section: `.forward()` and
`.backward()` compute every quantity by explicit matrix formula, matching
the derivation term for term.

Data convention: examples are stacked as **columns**, so a batch of $n$
examples with $n_0$ features is a $(n_0, n)$ array — matching the derivation
above exactly (no transposing between math and code).

In [ ]:
def relu(z):
    return np.maximum(0.0, z)


def relu_deriv(z):
    return (z > 0).astype(z.dtype)


def tanh_deriv(z):
    return 1.0 - np.tanh(z) ** 2


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1.0 - s)


ACTIVATIONS = {
    "relu": (relu, relu_deriv),
    "tanh": (np.tanh, tanh_deriv),
    "sigmoid": (sigmoid, sigmoid_deriv),
}


def softmax(z):
    # Subtract the per-column max for numerical stability; does not change
    # the result since softmax is shift-invariant.
    z = z - z.max(axis=0, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=0, keepdims=True)


class MLP:
    """An L-layer perceptron with hand-derived forward and backward passes.

    layer_sizes = [n0, n1, ..., nL]: input width, then each layer's width.
    The output layer (last one) always uses softmax; every earlier layer
    uses `activation`.
    """

    def __init__(self, layer_sizes, activation="relu", seed=0):
        rng = np.random.default_rng(seed)
        self.L = len(layer_sizes) - 1
        self.act, self.act_deriv = ACTIVATIONS[activation]
        self.W, self.b = [], []
        for l in range(self.L):
            fan_in, fan_out = layer_sizes[l], layer_sizes[l + 1]
            # He initialization: keeps activation variance stable across
            # layers regardless of which activation is used.
            scale = np.sqrt(2.0 / fan_in)
            self.W.append(rng.normal(scale=scale, size=(fan_out, fan_in)))
            self.b.append(np.zeros((fan_out, 1)))

    def forward(self, X):
        """X: (n0, n_examples). Returns a[L] and caches every z, a for backward()."""
        self.cache = {"a0": X}
        a = X
        for l in range(1, self.L + 1):
            z = self.W[l - 1] @ a + self.b[l - 1]
            a = softmax(z) if l == self.L else self.act(z)
            self.cache[f"z{l}"] = z
            self.cache[f"a{l}"] = a
        return a

    def backward(self, Y):
        """Y: (n_classes, n_examples) one-hot. Must be called after forward().
        Returns (grads_W, grads_b), one entry per layer, matching self.W/self.b."""
        n = Y.shape[1]
        grads_W, grads_b = [None] * self.L, [None] * self.L
        # Step 1: output-layer error signal (softmax + cross-entropy cancellation).
        delta = self.cache[f"a{self.L}"] - Y
        for l in range(self.L, 0, -1):
            a_prev = self.cache[f"a{l - 1}"]
            # Step 3: parameter gradients from this layer's error signal.
            grads_W[l - 1] = (delta @ a_prev.T) / n
            grads_b[l - 1] = delta.sum(axis=1, keepdims=True) / n
            if l > 1:
                # Step 2: propagate the error signal one layer upstream.
                z_prev = self.cache[f"z{l - 1}"]
                delta = (self.W[l - 1].T @ delta) * self.act_deriv(z_prev)
        return grads_W, grads_b

    def loss(self, X, Y, eps=1e-12):
        a = self.forward(X)
        return -np.mean(np.sum(Y * np.log(a + eps), axis=0))

    def predict(self, X):
        return self.forward(X).argmax(axis=0)


print("MLP class defined: .forward() and .backward() implement the boxed formulas above.")

## Gradient Checking

Deriving a gradient by hand is exactly the kind of step where a sign
error or a transposed matrix silently produces plausible-looking but wrong
numbers — training can even appear to "work" on a bug. The standard defense,
used in 1a for the single-neuron gradient, is a **numerical gradient check**:
perturb one parameter by a small $\epsilon$, measure how much the loss
actually changes, and compare that finite-difference estimate to the
analytic gradient our `.backward()` computed.

For a scalar parameter $p$, the **central difference** approximation is
$$
\frac{\partial J}{\partial p} \approx \frac{J(p+\epsilon) - J(p-\epsilon)}{2\epsilon}
$$
which is accurate to $O(\epsilon^2)$ (versus $O(\epsilon)$ for a one-sided
difference), so a small $\epsilon$ (we use $10^{-5}$) gives an extremely
tight numerical estimate. Checking *every* parameter this way would require
one full forward pass per parameter — too slow for a real network — so we
check a random handful of entries spread across every layer's weights and
biases. If those agree to many significant figures, the analytic
backward pass is almost certainly correct everywhere (a bug would have to
conspire to be invisible at every sampled entry, across every layer, which
is exceedingly unlikely).

One wrinkle: `gradient_check` works for *any* activation, since
`.backward()` only ever calls `self.act_deriv`, but ReLU's derivative is
genuinely discontinuous at $z=0$ (a "kink"), so a finite-difference estimate
that happens to straddle that exact point will legitimately disagree with
the one-sided analytic derivative there — that is a property of ReLU itself,
not a bug. We therefore check the network with a smooth activation (tanh),
which has no such discontinuity anywhere, so the check exercises exactly the
same `.backward()` code path (the formulas are activation-agnostic) without
this unrelated numerical wrinkle. The ReLU network we train below is
validated the direct way instead: it drives the loss down and accuracy up,
which a genuinely wrong gradient could not do.

In [ ]:
def gradient_check(model, X, Y, n_checks=30, eps=1e-5, seed=0):
    '''Compares model.backward()'s analytic gradients against central-difference
    numerical gradients at n_checks randomly chosen parameter entries, spread
    across every layer's W and b. Returns the list of per-parameter relative errors.
    '''
    rng = np.random.default_rng(seed)

    # One forward+backward pass gives the analytic gradient for every parameter.
    model.forward(X)
    grads_W, grads_b = model.backward(Y)

    # Flatten every parameter into a single list of (array, flat_index) targets
    # to sample from, so the check covers every layer, not just the first.
    targets = []
    for l in range(model.L):
        for flat_idx in range(model.W[l].size):
            targets.append(("W", l, flat_idx))
        for flat_idx in range(model.b[l].size):
            targets.append(("b", l, flat_idx))

    chosen = rng.choice(len(targets), size=min(n_checks, len(targets)), replace=False)

    rel_errors = []
    for t in chosen:
        kind, l, flat_idx = targets[t]
        param = model.W[l] if kind == "W" else model.b[l]
        analytic = (grads_W[l] if kind == "W" else grads_b[l]).flat[flat_idx]

        original = param.flat[flat_idx]
        param.flat[flat_idx] = original + eps
        loss_plus = model.loss(X, Y)
        param.flat[flat_idx] = original - eps
        loss_minus = model.loss(X, Y)
        param.flat[flat_idx] = original  # restore exactly

        numeric = (loss_plus - loss_minus) / (2 * eps)
        rel_err = abs(analytic - numeric) / max(abs(analytic), abs(numeric), 1e-8)
        rel_errors.append(rel_err)

    return np.array(rel_errors)


# A small toy network and a handful of random examples — big enough to
# exercise every layer, small enough that 30 finite-difference evaluations
# are instant. tanh (not ReLU) hidden activation: smooth everywhere, so the
# check is not confounded by ReLU's kink at z=0 (see note above). The
# .backward() code path exercised here is identical to the ReLU network below
# — only self.act_deriv's formula differs.
toy_net = MLP(layer_sizes=[8, 6, 5, 3], activation="tanh", seed=SEED)
X_toy = np.random.default_rng(SEED).normal(size=(8, 12))
labels_toy = np.random.default_rng(SEED + 1).integers(0, 3, size=12)
Y_toy = np.zeros((3, 12))
Y_toy[labels_toy, np.arange(12)] = 1.0

rel_errors = gradient_check(toy_net, X_toy, Y_toy, n_checks=30, eps=1e-5, seed=SEED)
print(f"gradient check over {len(rel_errors)} parameters (W and b, every layer):")
print(f"  max relative error:  {rel_errors.max():.3e}")
print(f"  mean relative error: {rel_errors.mean():.3e}")
assert rel_errors.max() < 1e-5, "analytic and numerical gradients should agree to high precision"
print("PASS: analytic backward() gradients match finite-difference gradients.")

The maximum relative error across every sampled parameter — spanning
every layer's weights and biases — is many orders of magnitude below the
$10^{-5}$ threshold, which is exactly the confirmation we need: the boxed
formulas from "Deriving Backpropagation" are correctly implemented in
`.backward()`, layer by layer, for this architecture. We can now trust
`.backward()`'s gradients to actually train the network.

## Activation Functions

The backward recursion $\delta^{[l]} = (W^{[l+1]})^\top\delta^{[l+1]}
\odot g'(z^{[l]})$ multiplies the propagated error by the activation
derivative $g'(z^{[l]})$ at *every* layer. This makes the *shape* of $g'$ —
not just $g$ — directly consequential for training: if $g'$ is close to
zero over some input range, any error signal passing through a unit whose
pre-activation lands in that range gets multiplied by (approximately) zero
and effectively stops propagating backward through it. This phenomenon is
called **saturation**.

In [ ]:
z = np.linspace(-6, 6, 400)

fns = {
    "sigmoid": (sigmoid(z), sigmoid_deriv(z)),
    "tanh": (np.tanh(z), tanh_deriv(z)),
    "relu": (relu(z), relu_deriv(z)),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
for ax, (name, (g, gprime)) in zip(axes, fns.items()):
    ax.plot(z, g, label=f"{name}(z)", linewidth=2)
    ax.plot(z, gprime, label=f"{name}'(z)", linewidth=2, linestyle="--")
    ax.axhline(0, color="gray", linewidth=0.6)
    ax.set_xlabel("z")
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle("Activation functions and their derivatives")
plt.tight_layout()
plt.show()

print("max derivative away from 0:")
for name, (_, gprime) in fns.items():
    far_from_zero = np.abs(z) > 4
    print(f"  {name:8s}: max|g'| for |z|>4 is {np.abs(gprime[far_from_zero]).max():.4f}"
          f", max|g'| overall is {np.abs(gprime).max():.4f}")

- **Sigmoid** saturates on *both* sides: $\sigma'(z) \to 0$ as
  $z \to \pm\infty$, and even by $|z|=4$ the derivative has collapsed to
  under 0.02. Any unit whose pre-activation is pushed strongly positive or
  strongly negative stops receiving a useful gradient — the classic
  **vanishing gradient** failure mode, and the main reason sigmoid is now
  rarely used in hidden layers of deep networks (it remains useful at an
  *output* layer for a single probability, as in 1a).
- **Tanh** has the same double-sided saturation as sigmoid (it is a rescaled
  sigmoid, $\tanh(z) = 2\sigma(2z)-1$), but its derivative peaks at 1
  instead of 0.25, so gradients shrink less aggressively per layer near
  $z=0$. It shares sigmoid's problem far from zero.
- **ReLU** saturates on only *one* side: for $z>0$, $\text{ReLU}'(z)=1$
  exactly, no matter how large $z$ gets — no saturation, no vanishing
  gradient, at all in the active region. For $z<0$ the derivative is exactly
  0 (a unit that is "off" passes back no gradient at all — the "dying ReLU"
  problem — but at least the *active* half of the input range never
  saturates). This asymmetry, not any exotic property, is the main reason
  ReLU became the default hidden-layer activation for deep networks: with
  many stacked layers, the repeated multiplication in the backward recursion
  by derivatives close to 1 (rather than close to 0) is what keeps gradient
  signal from vanishing as it propagates back through depth.

This is precisely why the MLP we train below uses ReLU in its hidden
layers.

### Training the from-scratch MLP on MNIST

We now put every piece together: load a subset of real MNIST digits, build
an `MLP([784, 64, 32, 10], activation="relu")` (784 input pixels, two ReLU
hidden layers, 10-class softmax output), and train it with the exact
`.forward()` / `.backward()` gradients derived and verified above — plain
mini-batch gradient descent, no framework, no autograd. `torchvision.datasets.MNIST`
downloads the dataset automatically the first time this cell runs, to a local
`data/` folder next to this notebook — this works identically in Colab and
locally. We subsample heavily (a couple thousand images) and cap epochs to
stay well inside the runtime budget on a CPU — a teaching example, not a
leaderboard run.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_full = torchvision.datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_full = torchvision.datasets.MNIST(root="data", train=False, download=True, transform=transform)


def subsample(dataset, n_total, seed):
    g = np.random.default_rng(seed)
    idx = g.permutation(len(dataset))[:n_total]
    images = torch.stack([dataset[i][0] for i in idx])            # (n, 1, 28, 28)
    X = images.reshape(len(idx), -1).numpy().astype(np.float64).T  # (784, n) columns = examples
    labels = dataset.targets.numpy()[idx].astype(int)
    Y = np.zeros((10, len(idx)))
    Y[labels, np.arange(len(idx))] = 1.0
    return X, Y, labels


N_TRAIN, N_TEST = 2000, 400
X_train, Y_train, labels_train = subsample(train_full, N_TRAIN, seed=SEED)
X_test, Y_test, labels_test = subsample(test_full, N_TEST, seed=SEED + 1)

print("X_train:", X_train.shape, " Y_train:", Y_train.shape)
print("X_test: ", X_test.shape, " Y_test: ", Y_test.shape)

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(X_train[:, i].reshape(28, 28), cmap="gray")
    ax.set_title(f"label: {labels_train[i]}", fontsize=9)
    ax.axis("off")
plt.suptitle("MNIST training samples")
plt.show()

In [ ]:
def accuracy(model, X, labels):
    preds = model.predict(X)
    return (preds == labels).mean()


def train_mlp(model, X, Y, labels, X_val, Y_val, labels_val, lr=0.3, epochs=60, batch_size=64, seed=0):
    g = np.random.default_rng(seed)
    n = X.shape[1]
    history = {"train_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        order = g.permutation(n)
        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]
            Xb, Yb = X[:, batch], Y[:, batch]
            model.forward(Xb)
            grads_W, grads_b = model.backward(Yb)
            for l in range(model.L):
                model.W[l] -= lr * grads_W[l]
                model.b[l] -= lr * grads_b[l]
        history["train_loss"].append(model.loss(X, Y))
        history["train_acc"].append(accuracy(model, X, labels))
        history["val_acc"].append(accuracy(model, X_val, labels_val))
    return history


mlp = MLP(layer_sizes=[784, 64, 32, 10], activation="relu", seed=SEED)
# Pixel values (already scaled to [0,1] by ToTensor) keep the ReLU/softmax
# gradients well-behaved at 784 input dimensions with no other change needed.
history = train_mlp(mlp, X_train, Y_train, labels_train, X_test, Y_test, labels_test,
                     lr=0.3, epochs=60, batch_size=64, seed=SEED)

print(f"final train loss: {history['train_loss'][-1]:.4f}")
print(f"final train accuracy: {history['train_acc'][-1]:.3f}")
print(f"final test accuracy:  {history['val_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history["train_loss"])
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("training cross-entropy loss")
axes[0].set_title("From-scratch MLP: training loss")
axes[0].grid(alpha=0.3)

axes[1].plot(history["train_acc"], label="train accuracy")
axes[1].plot(history["val_acc"], label="test accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_title("From-scratch MLP: accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

assert history["train_loss"][-1] < history["train_loss"][0] * 0.5, "loss should drop substantially"
assert history["val_acc"][-1] > 0.7, "a correctly-trained MLP should clear 70% on this MNIST subset"
print("Confirmed: loss falls sharply and test accuracy climbs well above chance (10%) "
      "— the hand-derived backward pass genuinely trains this network.")

The loss curve falls steadily and test accuracy climbs far above the
10% chance level for 10 classes, using *only* the `.forward()`/`.backward()`
implementation built from the boxed formulas in "Deriving Backpropagation" —
the same gradient-checked equations, now doing real work on real digits.

## Key Takeaways

- A **multilayer perceptron** stacks the single-neuron computation
  from 1a layer after layer: $z^{[l]} = W^{[l]}a^{[l-1]}+b^{[l]}$,
  $a^{[l]}=g^{[l]}(z^{[l]})$, with a softmax output layer generalizing 1a's
  sigmoid to multi-class prediction.
- **Backpropagation** is the chain rule applied systematically, layer by
  layer, backward from the output: $\delta^{[L]} = a^{[L]} - y$ (the
  softmax + cross-entropy cancellation, exactly analogous to 1a's sigmoid +
  BCE cancellation), then $\delta^{[l]} = (W^{[l+1]})^\top\delta^{[l+1]}
  \odot g'(z^{[l]})$ propagating the error signal upstream, giving parameter
  gradients $\nabla_{W^{[l]}}J = \delta^{[l]}(a^{[l-1]})^\top$ and
  $\nabla_{b^{[l]}}J = \delta^{[l]}$ at every layer.
- We implemented `.forward()` and `.backward()` **entirely in NumPy, with no
  autograd**, matching those boxed formulas term for term, and confirmed
  correctness with a **numerical finite-difference gradient check**
  (maximum relative error many orders of magnitude below the $10^{-5}$
  threshold, across every layer's weights and biases).
- That same from-scratch implementation **trained on real MNIST digits**,
  with loss falling and test accuracy climbing far above chance — proof the
  hand-derived gradients are not just numerically correct in isolation, they
  genuinely drive learning.
- **Sigmoid and tanh saturate on both sides** (their derivatives vanish for
  large $|z|$), which causes the **vanishing gradient** problem in deep
  networks — the backward recursion multiplies by $g'(z^{[l]})$ at every
  layer, so a chain of near-zero derivatives kills the gradient signal
  before it reaches early layers. **ReLU** saturates on only one side,
  which is why it is the default hidden-layer activation in modern deep
  networks.
- Everything derived and implemented here — the forward pass, backprop, and
  gradient checking — carries over unchanged when we rebuild this exact MLP
  in PyTorch and let autograd do the backward pass automatically (Lesson
  2b).